In [ ]:
import os
import cv2
import numpy as np
import zipfile

path = "/content/food (2).zip"
extract_path = "/content/food_dataset"

# Ensure the extract_path exists
os.makedirs(extract_path, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

X = []
y = []

# Determine the actual root for class folders
# It could be 'extract_path' itself, or 'extract_path/food' if there's a single top-level folder in the zip
base_dir_for_classes = extract_path
contents_of_extract_path = os.listdir(extract_path)

# If there's a single item in extract_path and it's a directory, assume it's the actual dataset root
if len(contents_of_extract_path) == 1 and os.path.isdir(os.path.join(extract_path, contents_of_extract_path[0])):
    base_dir_for_classes = os.path.join(extract_path, contents_of_extract_path[0])

# Iterate through each class folder
for class_name in os.listdir(base_dir_for_classes):
    class_folder_path = os.path.join(base_dir_for_classes, class_name)

    if os.path.isdir(class_folder_path): # Ensure it's a directory (i.e., a class folder)
        for img_file_name in os.listdir(class_folder_path):
            img_path = os.path.join(class_folder_path, img_file_name)

            # Check if it's a file and a common image extension
            if os.path.isfile(img_path) and img_file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                img = cv2.imread(img_path)

                if img is not None:
                    img = cv2.resize(img, (64,64))
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                    X.append(img.flatten())
                    y.append(class_name) # The class_name is the actual label

X = np.array(X)

print("Dataset Shape:", X.shape)
print("Labels:", len(y))

Dataset Shape: (9, 4096)
Labels: 9


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(7, 4096)
(2, 4096)


In [ ]:
from sklearn.svm import SVC

model = SVC(kernel='linear')

model.fit(X_train, y_train)

SVC(kernel='linear')

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.0


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      burger       0.00      0.00      0.00       1.0
       pizza       0.00      0.00      0.00       0.0
    sandwich       0.00      0.00      0.00       1.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
import cv2

# Correctly load an actual image file from the extracted dataset
# Using a sample image path from the dataset structure
img = cv2.imread("/content/food_dataset/food/sandwich/sandwich1.jpg")

# Check if the image was loaded successfully before processing
if img is not None:
    img = cv2.resize(img, (64,64))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = img.flatten().reshape(1,-1)

    prediction = model.predict(img)

    print("Predicted Food:", prediction[0])
else:
    print("Error: Could not load the image. Please check the image path.")

Predicted Food: sandwich


In [ ]:
calories = {
    "burger": 295,
    "pizza": 266,
    "sandwich": 250
}

food = prediction[0]

if food in calories:
    print("Estimated Calories:", calories[food], "kcal")
else:
    print("Calories not available")

Estimated Calories: 250 kcal
